# Ablation chain A (A1–A8): training and laboratory evaluation

This notebook trains ablation chain A of the two-stage cocoa bean grading framework (RetinaNet detector with differential-evolution (DE) anchors + Swin-Tiny classifier with EMDMFM/SAFM fusion, XGBoost head and isotonic calibration) on the MCC-02 dataset and evaluates each configuration on the laboratory test set.

Field evaluation of all twenty configurations (chains A, X and Y) is in `05_field_evaluation_and_analyses.ipynb`. For shared-backbone configurations (A4–A8) the detector and classifier checkpoints store different encoder states, because the classifier is trained after the detector on the same encoder; notebook 05 evaluates them with one encoder state for both heads, as deployed.

| ID | Change relative to previous configuration |
|---|---|
| A1 | Baseline: standalone Swin-Tiny + linear head, standard anchors, no CLAHE |
| A2 | DE-optimised anchors |
| A3 | CLAHE + bilateral filter preprocessing |
| A4 | Backbone shared between detector and classifier |
| A5 | EMDMFM fusion (without SAFM) |
| A6 | SAFM (full AdaptiveSwin-CNN) |
| A7 | XGBoost classification head (uncalibrated) |
| A8 | Isotonic calibration (full system) |

The notebook was run on Google Colab (GPU) with the dataset on Google Drive; adjust `BASE_DIR` to your environment.

In [ ]:
!pip install torchmetrics

## 1. Training and test-set evaluation (A1–A8)

Prepares ground-truth crops, computes the DE anchor configuration, trains the detectors and classifiers for each configuration (resumable from per-epoch checkpoints), fits the XGBoost head and isotonic calibration, and writes per-configuration metrics (mAP, accuracy, macro-F1, ECE, Brier) to JSON.

In [ ]:
# Ablation chain A (A1-A8) on the MCC-02 cocoa bean dataset.
# All models are trained from scratch; each configuration changes ONE variable
# relative to the previous one:
#
#   A1 : Swin-Tiny standalone + Linear head, standard anchors, no CLAHE
#   A2 : + DE-optimised anchors (standalone)          [anchor]
#   A3 : + CLAHE + bilateral filter (standalone)      [preprocessing]
#   A4 : + shared backbone (Linear, DE anchors, CLAHE) [backbone sharing]
#   A5 : + EMDMFM without SAFM (shared backbone)      [architecture]
#   A6 : + SAFM = full AdaptiveSwin-CNN (shared)      [architecture]
#   A7 : + XGBoost head, uncalibrated (reuses A6)     [classifier head]
#   A8 : + isotonic calibration = full system         [calibration]
#
# Reuse (the reused component is unaffected by the isolated variable):
#   A2 trains a new detector and reuses the A1 classifier (anchors do not
#     affect the GT-crop classifier).
#   A3 reuses the A2 detector (CLAHE does not affect detector training).
#   A7 and A8 reuse the A6 classifier and detector (XGBoost/calibration do
#     not touch the backbone or detector).
# Checkpoints are saved every epoch, so training can resume after a disconnect.

from google.colab import drive
drive.mount('/content/drive')

import os, json, copy, shutil, gc, warnings, time, pickle
import cv2
import numpy as np
import pandas as pd
import matplotlib; matplotlib.rcParams.update({'font.size': 12, 'figure.dpi': 150})
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageOps
from tqdm import tqdm
from collections import OrderedDict
from scipy.optimize import differential_evolution
from sklearn.metrics import (accuracy_score, f1_score, classification_report,
                             confusion_matrix, brier_score_loss)
from sklearn.utils.class_weight import compute_class_weight
from sklearn.calibration import CalibratedClassifierCV

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models.detection import RetinaNet
from torchvision.ops import FeaturePyramidNetwork, sigmoid_focal_loss
from torchvision.models.detection.anchor_utils import AnchorGenerator
from torchvision.models.detection.retinanet import RetinaNetClassificationHead
import timm
import xgboost as xgb
from torchmetrics.detection.mean_ap import MeanAveragePrecision

warnings.filterwarnings("ignore")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {DEVICE}")

# --- 1. Paths ---
BASE_DIR     = "/content/drive/MyDrive/UMK/Datasets/MCC-02-dataset"
RESULTS_DIR  = os.path.join(BASE_DIR, "results_final_ablation")
CKPT_DIR     = os.path.join(BASE_DIR, "checkpoints_final_ablation")
ABLATION_DIR = os.path.join(BASE_DIR, "ablation_results_final")
LOCAL_VM_DIR = "/content"
LOCAL_SPLIT_DIR = os.path.join(LOCAL_VM_DIR, "splits")
LOCAL_IMG_DIR   = os.path.join(LOCAL_VM_DIR, "images")
DRIVE_CROP_BKUP = os.path.join(BASE_DIR, "cropped_splits_gt_based_backup")
SHARED_CROP_DIR = os.path.join(LOCAL_VM_DIR, "crops_shared_gt_final")

for _d in [RESULTS_DIR, CKPT_DIR, ABLATION_DIR]:
    os.makedirs(_d, exist_ok=True)

ANCHOR_CONFIG_JSON = os.path.join(CKPT_DIR, "DE_anchor_config_final.json")

# --- 2. Data loading ---
DRIVE_JSON = os.path.join(BASE_DIR, "annotations_coco.json")
LOCAL_JSON = os.path.join(LOCAL_VM_DIR, "annotations_coco.json")
if not os.path.exists(LOCAL_JSON):
    shutil.copy2(DRIVE_JSON, LOCAL_JSON); print("annotations_coco.json copied")
if not os.path.exists(LOCAL_IMG_DIR):
    shutil.copytree(os.path.join(BASE_DIR, "images"), LOCAL_IMG_DIR); print("images/ copied")
if not os.path.exists(LOCAL_SPLIT_DIR):
    shutil.copytree(os.path.join(BASE_DIR, "splits"), LOCAL_SPLIT_DIR); print("splits/ copied")

with open(LOCAL_JSON) as f:
    coco_data = json.load(f)
cat_map        = {c['id']: c['name'] for c in coco_data['categories']}
sorted_cat_ids = sorted(cat_map.keys())
safe_label_map = {oid: si for si, oid in enumerate(sorted_cat_ids)}
class_names    = [cat_map[cid] for cid in sorted_cat_ids]
NUM_CLASSES    = len(class_names)
class_to_idx   = {n: i for i, n in enumerate(class_names)}
print(f"Classes: {class_names} (N={NUM_CLASSES})")

# --- 3. Ground-truth crops (shared by A1-A8; restored from a Drive backup for consistency) ---
def prepare_gt_crop():
    check = os.path.join(SHARED_CROP_DIR, 'test', class_names[0])
    if os.path.exists(check) and len(os.listdir(check)) > 0:
        print("GT crops already present on the VM -- skipping"); return
    dcheck = os.path.join(DRIVE_CROP_BKUP, 'test', class_names[0])
    if os.path.exists(dcheck) and len(os.listdir(dcheck)) > 0:
        print("Copying GT crops from the Drive backup...")
        shutil.copytree(DRIVE_CROP_BKUP, SHARED_CROP_DIR)
        print("Copy finished"); return
    print("Backup not found -- generating GT crops...")
    for sp in ['train', 'val', 'test']:
        for cls in class_names:
            os.makedirs(os.path.join(SHARED_CROP_DIR, sp, cls), exist_ok=True)
    fname_map = {}
    for sp in ['train', 'val', 'test']:
        for cls in class_names:
            d = os.path.join(LOCAL_SPLIT_DIR, sp, cls)
            if not os.path.exists(d): continue
            for fn in os.listdir(d):
                if fn.lower().endswith(('.jpg', '.png', '.jpeg')): fname_map[fn] = (sp, cls)
    img_anns = {}
    for ann in coco_data['annotations']:
        img_anns.setdefault(ann['image_id'], []).append(ann)
    n = 0
    for info in tqdm(coco_data['images'], desc="  GT Crop"):
        fn = info['file_name'].split('/')[-1]
        if fn not in fname_map: continue
        sp, cls = fname_map[fn]
        src = os.path.join(LOCAL_SPLIT_DIR, sp, cls, fn)
        if not os.path.exists(src): continue
        anns = img_anns.get(info['id'], [])
        if not anns: continue
        try:
            orig = ImageOps.exif_transpose(Image.open(src)).convert("RGB")
            iw, ih = orig.size
            for ci, ann in enumerate(anns):
                x, y, w, h = ann['bbox']
                x0, y0 = max(0, int(x)), max(0, int(y))
                x1, y1 = min(iw, int(x+w)), min(ih, int(y+h))
                if (x1-x0) < 10 or (y1-y0) < 10: continue
                orig.crop((x0, y0, x1, y1)).save(
                    os.path.join(SHARED_CROP_DIR, sp, cls, f"crop_{fn.split('.')[0]}_{ci}.jpg"))
                n += 1
        except Exception as e:
            print(f"  [WARN] {fn}: {e}")
    print(f"GT crop generation finished: {n} crops")

prepare_gt_crop()

# --- 4. Anchor configurations (DE is run once and reused for A2-A6; saved to JSON) ---
BASE_SIZES = [16, 24, 32, 48, 64]
STANDARD_SCALES, STANDARD_RATIOS = [1.0, 1.25, 1.5], [0.5, 1.0, 2.0]
STANDARD_GEN = AnchorGenerator(
    sizes=tuple(tuple(b*s for s in STANDARD_SCALES) for b in BASE_SIZES),
    aspect_ratios=tuple(tuple(STANDARD_RATIOS) for _ in BASE_SIZES))
print(f"Standard anchors (A1) num_anchors={STANDARD_GEN.num_anchors_per_location()[0]}")

if os.path.exists(ANCHOR_CONFIG_JSON):
    print("DE anchor config found -- loading it instead of recomputing")
    with open(ANCHOR_CONFIG_JSON) as f:
        _ac = json.load(f)
    OPT_SCALES, OPT_RATIOS = _ac["opt_scales"], _ac["opt_ratios"]
else:
    print("Running DE anchor optimisation (seed=42)...")
    _img_sizes   = np.array([[im['width'], im['height']] for im in coco_data['images']])
    median_img_w = float(np.median(_img_sizes[:, 0]))
    scale_factor = 224.0 / median_img_w
    _gt_raw = np.array([a['bbox'][2:4] for a in coco_data['annotations']])
    gt_w, gt_h = _gt_raw[:, 0]*scale_factor, _gt_raw[:, 1]*scale_factor
    base_size  = float(np.sqrt(np.median(gt_w*gt_h)))

    def _fitness(params):
        ratios, scales = params[:3], params[3:]
        aw, ah = [], []
        for r in ratios:
            for s in scales:
                aw.append(base_size*s*np.sqrt(r)); ah.append(base_size*s/np.sqrt(r))
        aw = np.array(aw)[None, :]; ah = np.array(ah)[None, :]
        gw_, gh_ = gt_w[:, None], gt_h[:, None]
        inter = np.minimum(gw_, aw)*np.minimum(gh_, ah)
        iou = inter/(gw_*gh_ + aw*ah - inter + 1e-9)
        return 1.0 - np.mean(np.max(iou, axis=1))

    _res = differential_evolution(_fitness, bounds=[(0.2,5.0)]*3 + [(0.5,3.0)]*3,
                                  strategy='best1bin', maxiter=100, popsize=30, seed=42)
    _sr = []
    for r in sorted(_res.x[:3]):
        _sr.append(r)
        if r != 0: _sr.append(1.0/r)
    OPT_RATIOS = sorted(list(set(_sr)))
    OPT_SCALES = sorted(_res.x[3:].tolist())
    with open(ANCHOR_CONFIG_JSON, 'w') as f:
        json.dump({"base_sizes": BASE_SIZES, "opt_scales": [float(x) for x in OPT_SCALES],
                   "opt_ratios": [float(x) for x in OPT_RATIOS],
                   "de_mean_max_iou": float(1-_res.fun)}, f, indent=2)
    print(f"DE finished. Mean max IoU={1-_res.fun:.4f}. Saved: {ANCHOR_CONFIG_JSON}")

DE_GEN = AnchorGenerator(
    sizes=tuple(tuple(b*s for s in OPT_SCALES) for b in BASE_SIZES),
    aspect_ratios=tuple(tuple(OPT_RATIOS) for _ in BASE_SIZES))
print(f"DE anchors (A2-A6) num_anchors={DE_GEN.num_anchors_per_location()[0]}")

# --- 5. Preprocessing ---
def apply_clahe_rgb(pil_img):
    n = np.array(pil_img)
    n = cv2.bilateralFilter(n, d=5, sigmaColor=50, sigmaSpace=50)
    lab = cv2.cvtColor(n, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    l = cv2.createCLAHE(clipLimit=1.5, tileGridSize=(4,4)).apply(l)
    return Image.fromarray(cv2.cvtColor(cv2.merge((l,a,b)), cv2.COLOR_LAB2RGB))

IMG_SIZE, BATCH, DET_EP, CLF_EP = 224, 16, 25, 15

train_tf_plain = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)), transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20), transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225])])
val_tf_plain = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)), transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225])])

# --- 6. Datasets ---
class DetDataset(Dataset):
    def __init__(self, coco_json, root, filter_dir=None):
        self.root = root; valid = None
        if filter_dir:
            valid = set()
            for cls in os.listdir(filter_dir):
                d = os.path.join(filter_dir, cls)
                if os.path.isdir(d):
                    for fn in os.listdir(d): valid.add(fn)
        self.images = [im for im in coco_json['images']
                      if valid is None or im['file_name'].split('/')[-1] in valid]
        self.anns = {}
        for ann in coco_json['annotations']:
            self.anns.setdefault(ann['image_id'], []).append(ann)
    def __len__(self): return len(self.images)
    def __getitem__(self, idx):
        info = self.images[idx]
        img = ImageOps.exif_transpose(Image.open(os.path.join(self.root, info['file_name']))).convert("RGB")
        img = transforms.ToTensor()(img)
        anns = self.anns.get(info['id'], [])
        boxes = [[a['bbox'][0], a['bbox'][1], a['bbox'][0]+a['bbox'][2], a['bbox'][1]+a['bbox'][3]] for a in anns]
        labels = [safe_label_map[a['category_id']]+1 for a in anns]
        if boxes:
            t = {"boxes": torch.as_tensor(boxes, dtype=torch.float32),
                 "labels": torch.as_tensor(labels, dtype=torch.int64)}
        else:
            t = {"boxes": torch.empty((0,4), dtype=torch.float32),
                 "labels": torch.empty((0,), dtype=torch.int64)}
        return img, t

class ClfDataset(Dataset):
    def __init__(self, df, tf=None, use_clahe=False):
        self.df = df; self.tf = tf; self.use_clahe = use_clahe
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = ImageOps.exif_transpose(Image.open(row['image_path'])).convert('RGB')
        if self.use_clahe: img = apply_clahe_rgb(img)
        if self.tf: img = self.tf(img)
        return img, int(row['label']), row['image_path']

def collate_fn(b): return tuple(zip(*b))

def make_loaders(crop_dir, use_clahe=False):
    def _df(split):
        data = []
        for cls in class_names:
            d = os.path.join(crop_dir, split, cls)
            if not os.path.exists(d): continue
            for fn in os.listdir(d):
                if fn.lower().endswith(('.jpg','.png','.jpeg')):
                    data.append({'image_path': os.path.join(d, fn), 'label': class_to_idx[cls]})
        return pd.DataFrame(data).reset_index(drop=True)
    tr, va, te = _df('train'), _df('val'), _df('test')
    print(f"  Dataset: train={len(tr)}, val={len(va)}, test={len(te)}")
    mkl = lambda df, sh, tf: DataLoader(ClfDataset(df, tf, use_clahe), batch_size=BATCH,
                                        shuffle=sh, pin_memory=True, num_workers=2,
                                        persistent_workers=True)
    return mkl(tr, True, train_tf_plain), mkl(va, False, val_tf_plain), mkl(te, False, val_tf_plain), tr, va, te

# --- 7. Training / evaluation functions ---
def train_detector(det_model, ckpt_path, extra_meta=None):
    ds = DetDataset(coco_data, LOCAL_VM_DIR)
    ld = DataLoader(ds, batch_size=64, shuffle=True, collate_fn=collate_fn,
                    num_workers=2, pin_memory=True, persistent_workers=True)
    opt = optim.SGD(det_model.parameters(), lr=0.002, momentum=0.9, weight_decay=5e-4)
    sch = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=DET_EP, eta_min=1e-5)
    start = 0
    if os.path.exists(ckpt_path):
        ck = torch.load(ckpt_path, map_location=DEVICE)
        det_model.load_state_dict(ck['model_state']); opt.load_state_dict(ck['optimizer_state'])
        if 'scheduler_state' in ck: sch.load_state_dict(ck['scheduler_state'])
        start = ck['epoch']+1
        print(f"  [Resume] Detector from epoch {start}/{DET_EP}")
    if start >= DET_EP:
        print("  Detector already trained. Skipping."); return det_model
    det_model.train()
    for ep in range(start, DET_EP):
        ls, nok, nnan = 0.0, 0, 0
        for imgs, tgts in tqdm(ld, desc=f"  Det {ep+1}/{DET_EP}"):
            imgs = [i.to(DEVICE) for i in imgs]
            tgts = [{k: v.to(DEVICE) for k, v in t.items()} for t in tgts]
            loss = sum(det_model(imgs, tgts).values())
            if not torch.isfinite(loss): nnan += 1; opt.zero_grad(); continue
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(det_model.parameters(), 10.0)
            opt.step(); ls += loss.item(); nok += 1
        sch.step(); avg = ls/max(1,nok)
        print(f"  Ep {ep+1:02d}/{DET_EP} Loss={avg:.4f} LR={opt.param_groups[0]['lr']:.2e} NaN={nnan}")
        payload = {'epoch': ep, 'model_state': det_model.state_dict(),
                  'optimizer_state': opt.state_dict(), 'scheduler_state': sch.state_dict(), 'loss': avg}
        if extra_meta: payload.update(extra_meta)
        torch.save(payload, ckpt_path)
    return det_model

def eval_map(det_model):
    te_ds = DetDataset(coco_data, LOCAL_VM_DIR, filter_dir=os.path.join(LOCAL_SPLIT_DIR, 'test'))
    te_ld = DataLoader(te_ds, batch_size=16, shuffle=False, collate_fn=collate_fn, num_workers=2)
    metric = MeanAveragePrecision(box_format='xyxy', iou_type='bbox')
    det_model.eval()
    with torch.no_grad():
        for imgs, tgts in tqdm(te_ld, desc="  mAP eval"):
            imgs = [i.to(DEVICE) for i in imgs]; outs = det_model(imgs)
            metric.update(
                [{"boxes": o["boxes"].cpu(), "scores": o["scores"].cpu(), "labels": o["labels"].cpu()} for o in outs],
                [{"boxes": t["boxes"].cpu(), "labels": t["labels"].cpu()} for t in tgts])
    r = metric.compute()
    m5095, m50, m75 = float(r['map'].item()), float(r['map_50'].item()), float(r['map_75'].item())
    print(f"  mAP@50={m50:.4f} mAP@75={m75:.4f} mAP@50:95={m5095:.4f}")
    return m5095, m50, m75

def train_classifier(clf_model, tr_ld, va_ld, ckpt_path):
    labels = [int(r['label']) for _, r in tr_ld.dataset.df.iterrows()]
    try:
        cw = torch.tensor(compute_class_weight('balanced', classes=np.unique(labels), y=labels),
                          dtype=torch.float).to(DEVICE)
    except Exception: cw = None
    crit = nn.CrossEntropyLoss(weight=cw, label_smoothing=0.1)
    opt = optim.AdamW(clf_model.parameters(), lr=1e-4, weight_decay=1e-4)
    sch = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=CLF_EP, eta_min=1e-6)
    scaler = torch.amp.GradScaler('cuda') if DEVICE.type == 'cuda' else None
    start, best_acc = 0, 0.0
    best_wts = copy.deepcopy(clf_model.state_dict())
    if os.path.exists(ckpt_path):
        ck = torch.load(ckpt_path, map_location=DEVICE)
        clf_model.load_state_dict(ck['model_state']); opt.load_state_dict(ck['optimizer_state'])
        if 'scheduler_state' in ck: sch.load_state_dict(ck['scheduler_state'])
        start, best_acc = ck['epoch']+1, ck.get('best_val_acc', 0.0)
        best_wts = copy.deepcopy(clf_model.state_dict())
        print(f"  [Resume] Classifier from epoch {start}/{CLF_EP} (best={best_acc:.4f})")
    if start >= CLF_EP:
        print("  Classifier already trained. Skipping."); clf_model.load_state_dict(best_wts)
        return clf_model, best_acc
    for ep in range(start, CLF_EP):
        clf_model.train(); tot, corr = 0, 0
        for imgs, lbls, _ in tqdm(tr_ld, desc=f"  Clf {ep+1}/{CLF_EP}"):
            imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE); opt.zero_grad()
            if scaler:
                with torch.amp.autocast('cuda'): out = clf_model(imgs); loss = crit(out, lbls)
                scaler.scale(loss).backward(); scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(clf_model.parameters(), 2.0)
                scaler.step(opt); scaler.update()
            else:
                out = clf_model(imgs); loss = crit(out, lbls); loss.backward()
                torch.nn.utils.clip_grad_norm_(clf_model.parameters(), 2.0); opt.step()
            corr += (out.argmax(1)==lbls).sum(); tot += imgs.size(0)
        sch.step()
        clf_model.eval(); vc, vt = 0, 0
        with torch.no_grad():
            for imgs, lbls, _ in va_ld:
                imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
                vc += (clf_model(imgs).argmax(1)==lbls).sum(); vt += imgs.size(0)
        va = float(vc/vt) if vt > 0 else 0.0
        print(f"  Ep {ep+1:02d}/{CLF_EP} Train={float(corr/tot):.4f} Val={va:.4f}")
        if va > best_acc: best_acc = va; best_wts = copy.deepcopy(clf_model.state_dict())
        torch.save({'epoch': ep, 'model_state': clf_model.state_dict(), 'optimizer_state': opt.state_dict(),
                   'scheduler_state': sch.state_dict(), 'best_val_acc': best_acc}, ckpt_path)
    clf_model.load_state_dict(best_wts)
    print(f"  Training finished. Best Val Acc = {best_acc:.4f}")
    return clf_model, best_acc

def compute_ece(y_true, y_proba, n_bins=15):
    conf = y_proba.max(axis=1); pred = y_proba.argmax(axis=1)
    acc = (pred==y_true).astype(float); bins = np.linspace(0,1,n_bins+1); ece = 0.0
    for i in range(n_bins):
        m = (conf>=bins[i]) & (conf<bins[i+1])
        if m.sum()==0: continue
        ece += (m.sum()/len(y_true)) * abs(acc[m].mean()-conf[m].mean())
    return ece

def eval_classifier(clf_model, te_ld, ablation_id=""):
    clf_model.eval(); yt, yp = [], []
    with torch.no_grad():
        for imgs, lbls, _ in tqdm(te_ld, desc=f"  Clf eval [{ablation_id}]"):
            yt.extend(lbls.numpy()); yp.extend(clf_model(imgs.to(DEVICE)).argmax(1).cpu().numpy())
    return _report(np.array(yt), np.array(yp), ablation_id, cmap='Blues')

def eval_from_labels(y_true, y_pred, ablation_id):
    return _report(y_true, y_pred, ablation_id, cmap='Greens')

def _report(yt, yp, ablation_id, cmap):
    acc = float(accuracy_score(yt, yp)); f1m = float(f1_score(yt, yp, average='macro'))
    rpt = classification_report(yt, yp, target_names=class_names, digits=4, output_dict=True)
    pc = {}
    for cls in class_names:
        d = rpt[cls]; s = int(d['support']); r = float(d['recall'])
        pc[cls] = {"precision": round(float(d['precision']),4), "recall": round(r,4),
                  "f1": round(float(d['f1-score']),4), "support": s, "missed": s-int(round(r*s))}
    print(f"\n  [{ablation_id}] Acc={acc:.4f} F1-macro={f1m:.4f}")
    for cls in class_names:
        d = pc[cls]
        print(f"  {cls:<15} P={d['precision']:.4f} R={d['recall']:.4f} F1={d['f1']:.4f} Miss={d['missed']}")
    cm = confusion_matrix(yt, yp)
    plt.figure(figsize=(8,6)); sns.heatmap(cm, annot=True, fmt='d', cmap=cmap,
        xticklabels=class_names, yticklabels=class_names)
    plt.title(f'Confusion Matrix -- {ablation_id}'); plt.tight_layout()
    p = os.path.join(ABLATION_DIR, f"confusion_matrix_{ablation_id}.png")
    plt.savefig(p, dpi=300, bbox_inches='tight'); plt.show()
    print(f"  Confusion matrix: {p}")
    return acc, f1m, pc

def save_result(ablation_id, metrics):
    p = os.path.join(ABLATION_DIR, f"{ablation_id}.json")
    with open(p, 'w') as f: json.dump({'id': ablation_id, **metrics}, f, indent=2)
    print(f"\n{'='*65}\n{ablation_id} -> {p}")
    for k, v in metrics.items():
        if k == 'per_class' or v is None: continue
        print(f"    {k:<20}: {v:.4f}" if isinstance(v, float) else f"    {k:<20}: {v}")
    print('='*65)

# --- 8. Model definitions ---
class TunedRetinaNetHead(RetinaNetClassificationHead):
    """RetinaNet classification head with focal loss alpha=0.5, gamma=2.0 (used for all detectors A1-A6)."""
    def __init__(self, *args, focal_alpha=0.5, focal_gamma=2.0, **kwargs):
        super().__init__(*args, **kwargs)
        self.focal_alpha = focal_alpha; self.focal_gamma = focal_gamma
    def compute_loss(self, targets, head_outputs, matched_idxs):
        losses = []; cls_logits = head_outputs["cls_logits"]
        for tgt, cl_p, mi_p in zip(targets, cls_logits, matched_idxs):
            fg = torch.where(mi_p >= 0)[0]; nf = fg.numel()
            gt_ = torch.zeros_like(cl_p); gt_[fg, tgt["labels"][mi_p[fg]]] = 1.0
            vi = mi_p != -2
            losses.append(sigmoid_focal_loss(cl_p[vi], gt_[vi], alpha=self.focal_alpha,
                                             gamma=self.focal_gamma, reduction="sum") / max(1, nf))
        return sum(losses) / len(targets)


class BasicSwinCLF(nn.Module):
    """A1, A2, A3, A4: Swin-Tiny + GAP + Linear."""
    def __init__(self, num_classes):
        super().__init__()
        self.swin = timm.create_model('swin_tiny_patch4_window7_224', pretrained=True,
                                      features_only=True, out_indices=(0,1,2))
        self.gap = nn.AdaptiveAvgPool2d(1); self.classifier = nn.Linear(384, num_classes)
    def _bchw(self, x):
        if x.dim()==4: return x if x.shape[1] in [96,192,384,768] else x.permute(0,3,1,2)
        B,L,C = x.shape; H = int(np.sqrt(L)); return x.permute(0,2,1).view(B,C,H,H)
    def forward(self, x):
        return self.classifier(self.gap(self._bchw(self.swin(x)[2])).flatten(1))


class SwinCNN_NoSAFM(nn.Module):
    """A5: EMDMFM without SAFM."""
    def __init__(self, num_classes):
        super().__init__()
        self.swin = timm.create_model('swin_tiny_patch4_window7_224', pretrained=True,
                                      features_only=True, out_indices=(0,1,2))
        self.em0=nn.Conv2d(96,128,1); self.em1=nn.Conv2d(192,128,1); self.em2=nn.Conv2d(384,128,1)
        self.dm2=nn.Sequential(nn.Conv2d(128,128,3,padding=1), nn.GELU())
        self.dm1=nn.Sequential(nn.Conv2d(128,128,3,padding=1), nn.GELU())
        self.dm0=nn.Sequential(nn.Conv2d(128,128,3,padding=1), nn.GELU())
        self.fm0=nn.Sequential(nn.Conv2d(256,128,1), nn.GELU())
        self.fm1=nn.Sequential(nn.Conv2d(256,128,1), nn.GELU())
        self.fm2=nn.Sequential(nn.Conv2d(256,128,1), nn.GELU())
        self.fm_agg=nn.Conv2d(384,256,1); self.gap=nn.AdaptiveAvgPool2d(1)
        self.fc_latent=nn.Linear(128*3+256,384); self.classifier=nn.Linear(384,num_classes)
    def _bchw(self, x):
        if x.dim()==4: return x if x.shape[1] in [96,192,384,768] else x.permute(0,3,1,2)
        B,L,C = x.shape; H = int(np.sqrt(L)); return x.permute(0,2,1).view(B,C,H,H)
    def get_latent_features(self, x):
        feats = self.swin(x)
        e0=self.em0(self._bchw(feats[0])); e1=self.em1(self._bchw(feats[1])); e2=self.em2(self._bchw(feats[2]))
        d2=self.dm2(e2)
        d1=self.dm1(F.interpolate(d2,size=e1.shape[2:],mode='bilinear',align_corners=False)+e1)
        d0=self.dm0(F.interpolate(d1,size=e0.shape[2:],mode='bilinear',align_corners=False)+e0)
        f2=self.fm2(torch.cat([e2,d2],1)); f1=self.fm1(torch.cat([e1,d1],1)); f0=self.fm0(torch.cat([e0,d0],1))
        fagg=self.fm_agg(torch.cat([f0,
            F.interpolate(f1,size=f0.shape[2:],mode='bilinear',align_corners=False),
            F.interpolate(f2,size=f0.shape[2:],mode='bilinear',align_corners=False)],1))
        latent = torch.cat([self.gap(f0).flatten(1), self.gap(f1).flatten(1),
                            self.gap(f2).flatten(1), self.gap(fagg).flatten(1)], dim=1)
        return F.relu(self.fc_latent(latent))
    def forward(self, x): return self.classifier(self.get_latent_features(x))


class AdaptiveSwinCNN_EMDMFM(nn.Module):
    """A6/A7/A8: EMDMFM + SAFM."""
    def __init__(self, num_classes=4):
        super().__init__()
        self.swin = timm.create_model('swin_tiny_patch4_window7_224', pretrained=True,
                                      features_only=True, out_indices=(0,1,2))
        self.em_proj0=nn.Conv2d(96,128,1); self.em_proj1=nn.Conv2d(192,128,1); self.em_proj2=nn.Conv2d(384,128,1)
        self.dm_conv2=nn.Sequential(nn.Conv2d(128,128,3,padding=1), nn.GELU())
        self.dm_conv1=nn.Sequential(nn.Conv2d(128,128,3,padding=1), nn.GELU())
        self.dm_conv0=nn.Sequential(nn.Conv2d(128,128,3,padding=1), nn.GELU())
        self.fm_conv0=nn.Sequential(nn.Conv2d(256,128,1), nn.GELU())
        self.fm_conv1=nn.Sequential(nn.Conv2d(256,128,1), nn.GELU())
        self.fm_conv2=nn.Sequential(nn.Conv2d(256,128,1), nn.GELU())
        self.fm_agg=nn.Conv2d(384,256,1); self.gap=nn.AdaptiveAvgPool2d(1)
        self.safm_attention=nn.MultiheadAttention(embed_dim=128, num_heads=8, batch_first=True)
        self.safm_norm=nn.LayerNorm(128); self.safm_dropout=nn.Dropout(0.1)
        self.fc_latent=nn.Linear(128*3+256,384); self.classifier=nn.Linear(384,num_classes)
    def _bchw(self, x):
        if x.dim()==4: return x if x.shape[1] in [96,192,384,768] else x.permute(0,3,1,2)
        B,L,C = x.shape; H = int(np.sqrt(L)); return x.permute(0,2,1).view(B,C,H,H)
    def get_latent_features(self, x):
        features = self.swin(x)
        e0=self.em_proj0(self._bchw(features[0])); e1=self.em_proj1(self._bchw(features[1])); e2=self.em_proj2(self._bchw(features[2]))
        d2=self.dm_conv2(e2)
        d1=self.dm_conv1(F.interpolate(d2,size=e1.shape[2:],mode='bilinear',align_corners=False)+e1)
        d0=self.dm_conv0(F.interpolate(d1,size=e0.shape[2:],mode='bilinear',align_corners=False)+e0)
        f2=self.fm_conv2(torch.cat([e2,d2],dim=1)); f1=self.fm_conv1(torch.cat([e1,d1],dim=1)); f0=self.fm_conv0(torch.cat([e0,d0],dim=1))
        f_agg=self.fm_agg(torch.cat([f0,
            F.interpolate(f1,size=f0.shape[2:],mode='bilinear',align_corners=False),
            F.interpolate(f2,size=f0.shape[2:],mode='bilinear',align_corners=False)],dim=1))
        f_agg_pool=self.gap(f_agg).flatten(1)
        tokens=torch.cat([self.gap(f).flatten(1).unsqueeze(1) for f in [f0,f1,f2]],dim=1)
        attn_out,_=self.safm_attention(tokens,tokens,tokens)
        latent=torch.cat([self.safm_norm(tokens+self.safm_dropout(attn_out)).flatten(1), f_agg_pool],dim=1)
        return F.relu(self.fc_latent(latent))
    def forward(self, x): return self.classifier(self.get_latent_features(x))


def _bchw_generic(x):
    if x.dim()==4: return x if x.shape[1] in [96,192,384,768] else x.permute(0,3,1,2)
    B,L,C = x.shape; H = int(np.sqrt(L)); return x.permute(0,2,1).view(B,C,H,H)

class StandaloneBackbone(nn.Module):
    """A1, A2: detector backbone with its own Swin instance (not shared with the classifier)."""
    def __init__(self):
        super().__init__()
        self.swin = timm.create_model('swin_tiny_patch4_window7_224', pretrained=True,
                                      features_only=True, out_indices=(0,1,2))
        self.out_channels = 256
        self.fpn = FeaturePyramidNetwork([96,192,384], 256)
        self.ds1 = nn.Conv2d(256,256,3,stride=2,padding=1); self.ds2 = nn.Conv2d(256,256,3,stride=2,padding=1)
    def forward(self, x):
        sf = self.swin(x)
        xd = OrderedDict([(str(i), _bchw_generic(sf[i])) for i in range(3)])
        fp = self.fpn(xd)
        out = OrderedDict([("0",fp["0"]), ("1",fp["1"]), ("2",fp["2"])])
        out["3"] = self.ds1(out["2"]); out["4"] = self.ds2(out["3"])
        return out

class SharedBackbone(nn.Module):
    """A4, A5, A6: self.swin = clf_model.swin -- the same object, so the detector and
    classifier literally share backbone weights (not a copy)."""
    def __init__(self, clf_model):
        super().__init__()
        self.swin = clf_model.swin
        self.out_channels = 256
        self.fpn = FeaturePyramidNetwork([96,192,384], 256)
        self.ds1 = nn.Conv2d(256,256,3,stride=2,padding=1); self.ds2 = nn.Conv2d(256,256,3,stride=2,padding=1)
    def forward(self, x):
        sf = self.swin(x)
        xd = OrderedDict([(str(i), _bchw_generic(sf[i])) for i in range(3)])
        fp = self.fpn(xd)
        out = OrderedDict([("0",fp["0"]), ("1",fp["1"]), ("2",fp["2"])])
        out["3"] = self.ds1(out["2"]); out["4"] = self.ds2(out["3"])
        return out

def build_standalone_detector(anchor_gen):
    nan = anchor_gen.num_anchors_per_location()[0]
    det = RetinaNet(backbone=StandaloneBackbone(), num_classes=NUM_CLASSES+1, anchor_generator=anchor_gen,
                    min_size=224, max_size=224, score_thresh=0.05, nms_thresh=0.5,
                    fg_iou_thresh=0.4, bg_iou_thresh=0.3).to(DEVICE)
    det.head.classification_head = TunedRetinaNetHead(256, nan, NUM_CLASSES+1, focal_alpha=0.5, focal_gamma=2.0).to(DEVICE)
    return det

def build_shared_detector(clf_model, anchor_gen):
    nan = anchor_gen.num_anchors_per_location()[0]
    det = RetinaNet(backbone=SharedBackbone(clf_model), num_classes=NUM_CLASSES+1, anchor_generator=anchor_gen,
                    min_size=224, max_size=224, score_thresh=0.05, nms_thresh=0.5,
                    fg_iou_thresh=0.4, bg_iou_thresh=0.3).to(DEVICE)
    det.head.classification_head = TunedRetinaNetHead(256, nan, NUM_CLASSES+1, focal_alpha=0.5, focal_gamma=2.0).to(DEVICE)
    return det

def extract_embeddings(clf_model, loader, desc):
    embs, labs = [], []
    clf_model.eval()
    with torch.no_grad():
        for imgs, lbls, _ in tqdm(loader, desc=desc):
            embs.append(clf_model.get_latent_features(imgs.to(DEVICE)).cpu().numpy())
            labs.extend(lbls.numpy())
    return np.vstack(embs), np.array(labs)

# --- 9. Checkpoint paths ---
CKPT = {
    "A1_det": os.path.join(CKPT_DIR, "A1_final_det.pth"),
    "A1_clf": os.path.join(CKPT_DIR, "A1_final_clf.pth"),
    "A2_det": os.path.join(CKPT_DIR, "A2_final_det.pth"),
    "A3_clf": os.path.join(CKPT_DIR, "A3_final_clf.pth"),
    "A4_det": os.path.join(CKPT_DIR, "A4_final_det.pth"),
    "A4_clf": os.path.join(CKPT_DIR, "A4_final_clf.pth"),
    "A5_det": os.path.join(CKPT_DIR, "A5_final_det.pth"),
    "A5_clf": os.path.join(CKPT_DIR, "A5_final_clf.pth"),
    "A6_det": os.path.join(CKPT_DIR, "A6_final_det.pth"),
    "A6_clf": os.path.join(CKPT_DIR, "A6_final_clf.pth"),
    "A7_xgb": os.path.join(CKPT_DIR, "A7_final_xgb_nocal.pkl"),
    "A8_xgb": os.path.join(CKPT_DIR, "A8_final_xgb_calibrated.pkl"),
}

# --- 10. A1 -> A2 -> A3 (standalone backbone; CLAHE only in A3) ---
tr_ld, va_ld, te_ld, _, _, _ = make_loaders(SHARED_CROP_DIR, use_clahe=False)

print(f"\n{'='*70}\n  A1: Standalone + Standard Anchor + Linear (no CLAHE)\n{'='*70}")
det_a1 = build_standalone_detector(STANDARD_GEN)
train_detector(det_a1, CKPT["A1_det"], {"id": "A1", "anchor": "standard"})
map5095_a1, map50_a1, map75_a1 = eval_map(det_a1)
clf_a1 = BasicSwinCLF(NUM_CLASSES).to(DEVICE)
clf_a1, bestacc_a1 = train_classifier(clf_a1, tr_ld, va_ld, CKPT["A1_clf"])
acc_a1, f1_a1, pc_a1 = eval_classifier(clf_a1, te_ld, "A1")
save_result("A1", {"map_5095": map5095_a1, "map_50": map50_a1, "map_75": map75_a1,
    "accuracy": acc_a1, "f1_macro": f1_a1, "best_val_acc": bestacc_a1, "per_class": pc_a1,
    "backbone": "standalone", "anchor": "standard", "use_clahe": False,
    "note": 'Baseline: standalone Swin-Tiny + linear head, standard anchors, no CLAHE.'})
gc.collect(); torch.cuda.empty_cache()

print(f"\n{'='*70}\n  A2: + DE Anchor (standalone) -- classifier reuse A1\n{'='*70}")
det_a2 = build_standalone_detector(DE_GEN)
train_detector(det_a2, CKPT["A2_det"], {"id": "A2", "anchor": "de_optimized"})
map5095_a2, map50_a2, map75_a2 = eval_map(det_a2)
print("  A2 classifier = A1 classifier (reused; anchors do not affect the GT-crop classifier)")
save_result("A2", {"map_5095": map5095_a2, "map_50": map50_a2, "map_75": map75_a2,
    "accuracy": acc_a1, "f1_macro": f1_a1, "best_val_acc": bestacc_a1, "per_class": pc_a1,
    "backbone": "standalone", "anchor": "de_optimized", "use_clahe": False,
    "note": '+ DE anchors. Classifier identical to A1 (reused). Variable isolated: anchors.'})
gc.collect(); torch.cuda.empty_cache()

print(f"\n{'='*70}\n  A3: + CLAHE + Bilateral Filter -- detector reuse A2\n{'='*70}")
print("  A3 detector = A2 detector (reused; CLAHE does not affect detector training)")
tr_ld_c, va_ld_c, te_ld_c, _, _, _ = make_loaders(SHARED_CROP_DIR, use_clahe=True)
clf_a3 = BasicSwinCLF(NUM_CLASSES).to(DEVICE)
clf_a3, bestacc_a3 = train_classifier(clf_a3, tr_ld_c, va_ld_c, CKPT["A3_clf"])
acc_a3, f1_a3, pc_a3 = eval_classifier(clf_a3, te_ld_c, "A3")
save_result("A3", {"map_5095": map5095_a2, "map_50": map50_a2, "map_75": map75_a2,
    "accuracy": acc_a3, "f1_macro": f1_a3, "best_val_acc": bestacc_a3, "per_class": pc_a3,
    "backbone": "standalone", "anchor": "de_optimized", "use_clahe": True,
    "note": '+ CLAHE + bilateral filter. Detector identical to A2 (reused). Variable isolated: preprocessing.'})
gc.collect(); torch.cuda.empty_cache()

# --- 11. A4 -> A5 -> A6 (shared backbone, full retraining each time, with CLAHE) ---
print(f"\n{'='*70}\n  A4: + Shared Backbone (Linear, DE anchor, CLAHE) -- full retraining\n{'='*70}")
clf_a4 = BasicSwinCLF(NUM_CLASSES).to(DEVICE)
det_a4 = build_shared_detector(clf_a4, DE_GEN)
train_detector(det_a4, CKPT["A4_det"], {"id": "A4", "backbone": "shared"})
map5095_a4, map50_a4, map75_a4 = eval_map(det_a4)
clf_a4, bestacc_a4 = train_classifier(clf_a4, tr_ld_c, va_ld_c, CKPT["A4_clf"])
acc_a4, f1_a4, pc_a4 = eval_classifier(clf_a4, te_ld_c, "A4")
save_result("A4", {"map_5095": map5095_a4, "map_50": map50_a4, "map_75": map75_a4,
    "accuracy": acc_a4, "f1_macro": f1_a4, "best_val_acc": bestacc_a4, "per_class": pc_a4,
    "backbone": "shared", "anchor": "de_optimized", "use_clahe": True,
    "note": ('+ Shared backbone (linear head). Fully retrained: a shared backbone cannot reuse the Swin weights of A1-A3. Variable isolated: backbone sharing.')})
gc.collect(); torch.cuda.empty_cache()

print(f"\n{'='*70}\n  A5: + EMDMFM without SAFM (shared backbone) -- full retraining\n{'='*70}")
clf_a5 = SwinCNN_NoSAFM(NUM_CLASSES).to(DEVICE)
det_a5 = build_shared_detector(clf_a5, DE_GEN)
train_detector(det_a5, CKPT["A5_det"], {"id": "A5", "backbone": "shared"})
map5095_a5, map50_a5, map75_a5 = eval_map(det_a5)
clf_a5, bestacc_a5 = train_classifier(clf_a5, tr_ld_c, va_ld_c, CKPT["A5_clf"])
acc_a5, f1_a5, pc_a5 = eval_classifier(clf_a5, te_ld_c, "A5")
save_result("A5", {"map_5095": map5095_a5, "map_50": map50_a5, "map_75": map75_a5,
    "accuracy": acc_a5, "f1_macro": f1_a5, "best_val_acc": bestacc_a5, "per_class": pc_a5,
    "backbone": "shared", "anchor": "de_optimized", "use_clahe": True,
    "note": '+ EMDMFM without SAFM. Fully retrained. Variable isolated: architecture (EMDMFM).'})
gc.collect(); torch.cuda.empty_cache()

print(f"\n{'='*70}\n  A6: + SAFM = full AdaptiveSwin-CNN (shared backbone) -- full retraining\n{'='*70}")
clf_a6 = AdaptiveSwinCNN_EMDMFM(NUM_CLASSES).to(DEVICE)
det_a6 = build_shared_detector(clf_a6, DE_GEN)
train_detector(det_a6, CKPT["A6_det"], {"id": "A6", "backbone": "shared"})
map5095_a6, map50_a6, map75_a6 = eval_map(det_a6)
clf_a6, bestacc_a6 = train_classifier(clf_a6, tr_ld_c, va_ld_c, CKPT["A6_clf"])
acc_a6, f1_a6, pc_a6 = eval_classifier(clf_a6, te_ld_c, "A6")
save_result("A6", {"map_5095": map5095_a6, "map_50": map50_a6, "map_75": map75_a6,
    "accuracy": acc_a6, "f1_macro": f1_a6, "best_val_acc": bestacc_a6, "per_class": pc_a6,
    "backbone": "shared", "anchor": "de_optimized", "use_clahe": True,
    "note": '+ SAFM (full AdaptiveSwin-CNN). Fully retrained. Variable isolated: architecture (SAFM).'})
gc.collect(); torch.cuda.empty_cache()

# --- 12. A7 -> A8 (reuse A6; XGBoost and calibration do not touch the backbone) ---
print(f"\n{'='*70}\n  A7: + XGBoost, uncalibrated -- classifier+detector reused from A6\n{'='*70}")
X_train, y_train = extract_embeddings(clf_a6, tr_ld_c, "Features Train")
X_val,   y_val   = extract_embeddings(clf_a6, va_ld_c, "Features Val")
X_test,  y_test  = extract_embeddings(clf_a6, te_ld_c, "Features Test")

xgb_a7 = xgb.XGBClassifier(n_estimators=500, max_depth=6, learning_rate=0.05,
    objective='multi:softprob', num_class=NUM_CLASSES, tree_method='hist',
    device='cuda' if torch.cuda.is_available() else 'cpu', random_state=42)
xgb_a7.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
proba_raw = xgb_a7.predict_proba(X_test); y_pred_raw = proba_raw.argmax(axis=1)
ece_a7 = compute_ece(y_test, proba_raw)
brier_a7 = np.mean([brier_score_loss(y_test==c, proba_raw[:,c]) for c in range(NUM_CLASSES)])
with open(CKPT["A7_xgb"], 'wb') as f: pickle.dump(xgb_a7, f)
print(f"  ECE(raw)={ece_a7:.4f} Brier={brier_a7:.4f}")
acc_a7, f1_a7, pc_a7 = eval_from_labels(y_test, y_pred_raw, "A7")
save_result("A7", {"map_5095": map5095_a6, "map_50": map50_a6, "map_75": map75_a6,
    "accuracy": acc_a7, "f1_macro": f1_a7, "ece": float(ece_a7), "brier": float(brier_a7),
    "best_val_acc": bestacc_a6, "per_class": pc_a7, "backbone": "shared",
    "anchor": "de_optimized", "use_clahe": True,
    "note": '+ XGBoost (uncalibrated). Classifier and detector identical to A6 (reused). Variable isolated: classifier head.'})
gc.collect(); torch.cuda.empty_cache()

print(f"\n{'='*70}\n  A8: + Isotonic Calibration = full system -- XGBoost reused from A7\n{'='*70}")
xgb_a8_cal = CalibratedClassifierCV(xgb_a7, method='isotonic', cv='prefit')
xgb_a8_cal.fit(X_val, y_val)
proba_cal = xgb_a8_cal.predict_proba(X_test); y_pred_cal = proba_cal.argmax(axis=1)
ece_a8 = compute_ece(y_test, proba_cal)
brier_a8 = np.mean([brier_score_loss(y_test==c, proba_cal[:,c]) for c in range(NUM_CLASSES)])
with open(CKPT["A8_xgb"], 'wb') as f: pickle.dump(xgb_a8_cal, f)
print(f"  ECE: {ece_a7:.4f} -> {ece_a8:.4f} (reduction {(ece_a7-ece_a8)/ece_a7*100:.1f}%)")
acc_a8, f1_a8, pc_a8 = eval_from_labels(y_test, y_pred_cal, "A8")
save_result("A8", {"map_5095": map5095_a6, "map_50": map50_a6, "map_75": map75_a6,
    "accuracy": acc_a8, "f1_macro": f1_a8, "ece": float(ece_a8), "brier": float(brier_a8),
    "ece_before_calibration": float(ece_a7), "brier_before_calibration": float(brier_a7),
    "best_val_acc": bestacc_a6, "per_class": pc_a8, "backbone": "shared",
    "anchor": "de_optimized", "use_clahe": True,
    "note": ('+ Isotonic calibration (full pipeline). XGBoost identical to A7, calibrated here. Variable isolated: calibration.')})
gc.collect(); torch.cuda.empty_cache()

# --- 13. Final summary table A1-A8 ---
def print_final_table():
    rows = [
        ("A1", "Standalone + Standard Anchor + Linear"),
        ("A2", "+ DE Anchor"),
        ("A3", "+ CLAHE + Bilateral Filter"),
        ("A4", "+ Shared Backbone"),
        ("A5", "+ EMDMFM (no SAFM)"),
        ("A6", "+ SAFM = AdaptiveSwin-CNN"),
        ("A7", "+ XGBoost (no calibration)"),
        ("A8", "+ Isotonic Calibration (FULL) *"),
    ]
    print("\n" + "="*110)
    print("  FINAL SUMMARY -- ABLATION STUDY A1-A8")
    print("="*110)
    hdr = f"  {'ID':<3}{'Configuration':<38}{'mAP50':>8}{'mAP':>8}{'Acc':>8}{'F1':>8}{'ECE':>8}{'Broken_R':>12}{'Moldy_R':>12}"
    print(hdr); print("  " + "-"*106)
    for aid, name in rows:
        p = os.path.join(ABLATION_DIR, f"{aid}.json")
        if not os.path.exists(p):
            print(f"  {aid:<3}{name:<38}{'—':>8}{'—':>8}{'—':>8}{'—':>8}{'—':>8}{'—':>12}{'—':>12}")
            continue
        with open(p) as f: d = json.load(f)
        pc = d.get('per_class', {})
        def _r(cls):
            c = pc.get(cls, {})
            return f"{c.get('recall',0):.4f}(-{c.get('missed','?')})" if c else "—"
        ece = f"{d['ece']:.4f}" if d.get('ece') is not None else "—"
        print(f"  {aid:<3}{name:<38}{d['map_50']:>8.4f}{d['map_5095']:>8.4f}"
              f"{d['accuracy']:>8.4f}{d['f1_macro']:>8.4f}{ece:>8}{_r('Broken'):>12}{_r('Moldy'):>12}")
    print("="*110)
    print("  * = proposed method")

print_final_table()
print("\nDone. Checkpoints and JSON results are in:")
print(f"  Checkpoint : {CKPT_DIR}")
print(f"  JSON results: {ABLATION_DIR}")
print(f"  Anchor DE  : {ANCHOR_CONFIG_JSON}")